In [11]:
import yt_dlp
import shutil
import os

def download_youtube(url: str, audio_only: bool = True, output_dir: str = "downloads") -> str:
    os.makedirs(output_dir, exist_ok=True)
    out_template = os.path.join(output_dir, "%(title)s.%(ext)s")
    ffmpeg_path = shutil.which("ffmpeg")
    print(f"Using FFmpeg from: {ffmpeg_path}")
    ydl_opts = {
        'outtmpl': out_template,
        'quiet': False,
        'noplaylist': True
    }

    if audio_only:
        ydl_opts.update({
            'format': 'bestaudio/best',
            'ffmpeg_location': r'C:\ProgramData\chocoportable\bin',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'mp3',
                'preferredquality': '192'
            }]
        })
    else:
        ydl_opts.update({'format': 'bestvideo+bestaudio/best'})

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(url, download=True)
        downloaded_file = ydl.prepare_filename(info)

        if audio_only:
            downloaded_file = os.path.splitext(downloaded_file)[0] + ".mp3"

    print(f"✅ Downloaded file: {downloaded_file}")
    return downloaded_file

# # Example usage:
# file_path = download_youtube("https://www.youtube.com/watch?v=abc123xyz", audio_only=True)


In [12]:
url="https://www.youtube.com/watch?v=akF2HpS7L5g"

In [13]:
file=download_youtube(url, audio_only=True)

Using FFmpeg from: None
[youtube] Extracting URL: https://www.youtube.com/watch?v=akF2HpS7L5g
[youtube] akF2HpS7L5g: Downloading webpage
[youtube] akF2HpS7L5g: Downloading android sdkless player API JSON
[youtube] akF2HpS7L5g: Downloading tv client config
[youtube] akF2HpS7L5g: Downloading tv player API JSON
[youtube] akF2HpS7L5g: Downloading web safari player API JSON


[youtube] akF2HpS7L5g: Downloading m3u8 information
[info] akF2HpS7L5g: Downloading 1 format(s): 251
[download] downloads\Dr. Jyoti Bala Sharma on Rising Stroke Risks Among Young Adults at Fortis Noida.webm has already been downloaded
[download] 100% of    1.88MiB
[ExtractAudio] Destination: downloads\Dr. Jyoti Bala Sharma on Rising Stroke Risks Among Young Adults at Fortis Noida.mp3
Deleting original file downloads\Dr. Jyoti Bala Sharma on Rising Stroke Risks Among Young Adults at Fortis Noida.webm (pass -k to keep)
✅ Downloaded file: downloads\Dr. Jyoti Bala Sharma on Rising Stroke Risks Among Young Adults at Fortis Noida.mp3


In [14]:
import boto3

def upload_to_s3(file_path: str, bucket_name: str, s3_key: str):
    s3 = boto3.client('s3')
    s3.upload_file(file_path, bucket_name, s3_key)
    print(f"✅ Uploaded to: s3://{bucket_name}/{s3_key}")
    return f"s3://{bucket_name}/{s3_key}"

In [15]:
# Example
bucket_name = "fhl-transcribe-inputs"
s3_key = os.path.basename(file)
s3_uri = upload_to_s3(file, bucket_name, s3_key)

✅ Uploaded to: s3://fhl-transcribe-inputs/Dr. Jyoti Bala Sharma on Rising Stroke Risks Among Young Adults at Fortis Noida.mp3


In [16]:
import time

def transcribe_audio(job_name: str, s3_uri: str, output_bucket: str):
    transcribe = boto3.client('transcribe')
    media_format = s3_uri.split('.')[-1]  # auto-detect mp3/mp4/wav

    transcribe.start_transcription_job(
        TranscriptionJobName=job_name,
        Media={'MediaFileUri': s3_uri},
        MediaFormat=media_format,
        LanguageCode='en-US',
        OutputBucketName=output_bucket
    )

    while True:
        status = transcribe.get_transcription_job(TranscriptionJobName=job_name)
        state = status['TranscriptionJob']['TranscriptionJobStatus']
        if state in ['COMPLETED', 'FAILED']:
            break
        print("⌛ Waiting for transcription to complete...")
        time.sleep(15)

    if state == 'COMPLETED':
        uri = status['TranscriptionJob']['Transcript']['TranscriptFileUri']
        print(f"✅ Transcription completed.\nTranscript URL: {uri}")
    else:
        print("❌ Transcription failed!")


In [17]:
# Example usage:
transcribe_audio("youtube_transcription_job", s3_uri, output_bucket="fhl-transcribe-inputs")

ClientError: An error occurred (SubscriptionRequiredException) when calling the StartTranscriptionJob operation: The AWS Access Key Id needs a subscription for the service